In [4]:
import json

data = json.load(open('codemi.json', 'r'))
print(len(data))
print(data[0].keys())

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [2]:
print(data[0]["model"])

codegemma-2b


In [ ]:
import os
from contextlib import contextmanager
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_paths = ["../models/shadow_model_1_finetuned", "./model/shadow_model_2_finetuned", "./model/target_model_finetuned"]

keywords = {'', '(', ')', '{', '}', '_', "auto", "break", "case", "char", "const", "continue", "default", "do",
            "double", "else", "enum", "extern", "float", "for", "goto", "if",
            "inline", "int", "long", "register", "restrict", "return", "short",
            "signed", "sizeof", "static", "struct", "switch", "typedef", "union",
            "unsigned", "void", "volatile", "while", "_Alignas", "_Alignof",
            "_Atomic", "_Bool", "_Complex", "_Generic", "_Imaginary", "_Noreturn",
            "_Static_assert", "_Thread_local"}


MODEL_CACHE = {}

@contextmanager
def clear_cuda_cache():
    try:
        yield
    finally:
        torch.cuda.empty_cache()

def load_model(model_name, device='cuda'):
    if model_name in MODEL_CACHE:
        return MODEL_CACHE[model_name]

    tokenizer_path = f"../models/{model_name}"
    model_path = f"../models/{model_name}"

    tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.float16,
        device_map="auto"
    )

    checkpoints = {
        "codegemma-2b": 6000,
        "deepseek-coder-1.3b-instruct": 7000,
        "Qwen2.5-Coder-3B-Instruct": 4000
    }
    model_id = f"../models/{model_name}-lora/checkpoint-{checkpoints[model_name]}"
    p_model = PeftModel.from_pretrained(model, model_id)

    MODEL_CACHE[model_name] = (tokenizer, p_model)

    return tokenizer, p_model

def calculate_ranks(model_name, text, device='cuda:1'):

    with clear_cuda_cache(), torch.no_grad():

        tokenizer, p_model = load_model(model_name, device)

        encodings = tokenizer(text, return_tensors='pt')
        input_ids = encodings.input_ids.to(device)

        ranks = []

        tokens = input_ids[0].tolist()
        cleaned_tokens = [tokenizer.convert_ids_to_tokens(token).replace('Ġ', '').replace('Ċ', '') for token in tokens]

        seq_len = input_ids.shape[1]  

        past_key_values = None 

        for i in range(1, seq_len):

            current_input_ids = input_ids[:, :i]

            with torch.no_grad():

                if past_key_values is None:
                    outputs = p_model(current_input_ids)
                else:
                    outputs = p_model(input_ids[:, i - 1].unsqueeze(0), past_key_values=past_key_values)

                logits = outputs.logits 
                past_key_values = outputs.past_key_values 

            token = cleaned_tokens[i] 
            real_token_id = input_ids[0, i]  

          
            if token in keywords:
                continue

            probs = logits[0, -1, :]
            rank = torch.sum(probs > probs[real_token_id]).item() 
            

            ranks.append(rank)

       
        del input_ids, outputs, encodings

        return ranks

from tqdm import tqdm
for i in tqdm(range(len(data))):
    if "ranks" not in data[i] and data[i]["index"]==1:
        data[i]["ranks"] = calculate_ranks(data[i]["model"],data[i]["code"])
    else:
        continue
with open('codemi.json', 'w', encoding='utf-8') as file:
    json.dump(data, file, indent=4, ensure_ascii=False)

  0%|          | 0/41745 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

 83%|████████▎ | 34542/41745 [1:30:34<18:53,  6.36it/s]   


KeyboardInterrupt: 

In [2]:
import json
with open('codemi.json', 'w', encoding='utf-8') as file:
    json.dump(data, file, indent=4, ensure_ascii=False)

NameError: name 'data' is not defined

In [1]:
import json

data = json.load(open('codemi.json', 'r'))
count=0
for i in range(len(data)):
    if "ranks" in data[i]:
        count+=1
print(count)

9871


In [4]:
length = []
for i in range(len(data)):
    if "ranks" in data[i] and len(data[i]["ranks"])not in length:
        length.append(len(data[i]["ranks"]))
print(length)

[206, 248, 183, 177, 291, 201, 228, 154, 178, 185, 283, 150, 213, 151, 118, 127, 144, 259, 225, 334, 350, 205, 157, 102, 130, 215, 182, 152, 87, 170, 218, 175, 624, 310, 239, 142, 165, 254, 299, 222, 190, 258, 320, 73, 249, 181, 136, 204, 224, 134, 71, 238, 101, 171, 211, 195, 147, 189, 223, 256, 344, 513, 324, 265, 230, 323, 250, 379, 199, 169, 452, 594, 162, 96, 216, 143, 277, 407, 321, 208, 188, 193, 433, 346, 347, 200, 202, 269, 540, 274, 369, 296, 418, 360, 329, 232, 521, 303, 356, 333, 311, 217, 330, 613, 263, 306, 401, 229, 861, 394, 231, 210, 341, 361, 153, 186, 155, 655, 209, 345, 445, 339, 267, 327, 544, 146, 363, 457, 234, 271, 196, 251, 281, 149, 197, 156, 107, 117, 173, 133, 138, 187, 385, 331, 131, 314, 354, 262, 233, 294, 145, 497, 315, 275, 377, 432, 585, 159, 272, 368, 404, 328, 244, 530, 387, 302, 382, 313, 555, 289, 411, 510, 359, 282, 353, 203, 674, 447, 268, 808, 386, 242, 326, 297, 605, 337, 519, 180, 383, 338, 245, 168, 293, 137, 219, 184, 384, 167, 237, 266, 75,

In [5]:
import numpy as np
def rank_to_bin(ranks, max_rank=128):
    bin_vector = np.zeros(max_rank)
    for rank in ranks:
        if rank < max_rank:
            bin_vector[rank] += 1  # 将排名映射为bin的频次
    print("bin")
    print(bin_vector)
    return bin_vector


for i in range(len(data)):
    if data[i]["index"]==1:
        data[i]["bin"] = rank_to_bin(data[i]["ranks"])


length = []
for i in range(len(data)):
    if "bin" in data[i] and len(data[i]["bin"])not in length:
        length.append(len(data[i]["bin"]))
print(length)

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



[128]


In [ ]:

import json
from collections import defaultdict
import numpy as np
import torch.nn as nn
import torch
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import precision_score, recall_score, f1_score, matthews_corrcoef, confusion_matrix, roc_auc_score
from torch.utils.data import Dataset, DataLoader, random_split, Subset
import pandas as pd

# 定义模型类
class ImprovedModel(nn.Module):
    def __init__(self, input_dim):
        super(ImprovedModel, self).__init__()
        self.fc1 = nn.Linear(input_dim, 128) 
        self.bn1 = nn.BatchNorm1d(128)
        self.dropout1 = nn.Dropout(0.5)

        self.fc2 = nn.Linear(128, 64)
        self.bn2 = nn.BatchNorm1d(64)
        self.dropout2 = nn.Dropout(0.3)
        self.fc3 = nn.Linear(64, 2)  

    def forward(self, x):
        x = F.relu(self.bn1(self.fc1(x)))
        x = self.dropout1(x)
        x = F.relu(self.bn2(self.fc2(x)))
        x = self.dropout2(x)
        x = self.fc3(x)
        return x


class VecDataset(Dataset):
    def __init__(self, data):
        self.data = data
        self.classes = sorted(list(set(item['class'] for item in data)))
        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}
        self.models = sorted(list(set(item['model'] for item in data)))
        self.benchmarks = sorted(list(set(item['benchmark'] for item in data)))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        features = torch.tensor(item['bin'], dtype=torch.float32)
        label = self.class_to_idx[item['class']]
        return features, label


final_data = []
for i in range(len(data)):
    if data[i]["index"]==1:
        final_data.append(data[i])



full_dataset = VecDataset(final_data)

train_size = int(0.5 * len(full_dataset))
val_size = int(0.15 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size
train_dataset, val_dataset, test_dataset = random_split(full_dataset, [train_size, val_size, test_size])

In [ ]:
all_run_results=[]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


for run in range(5):
    print(f"\n=== Starting Run {run+1}/5 ===")

    input_dim = len(final_data[0]['bin'])


    train_loader = DataLoader(train_dataset, batch_size=100, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=100, shuffle=False)


    model = ImprovedModel(input_dim).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=3)


    best_val_loss = float('inf')
    patience = 5
    no_improve = 0
    num_epochs = 5

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        for batch_features, batch_labels in train_loader:
            batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
            optimizer.zero_grad()
            outputs = model(batch_features)
            loss = criterion(outputs, batch_labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()


        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        with torch.no_grad():
            for features, labels in val_loader:
                features, labels = features.to(device), labels.to(device)
                outputs = model(features)
                val_loss += criterion(outputs, labels).item()
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        avg_train_loss = train_loss/len(train_loader)
        avg_val_loss = val_loss/len(val_loader)
        val_accuracy = 100 * correct / total

        print(f'Epoch [{epoch+1}/{num_epochs}], '
              f'Train Loss: {avg_train_loss:.4f}, '
              f'Val Loss: {avg_val_loss:.4f}, '
              f'Val Acc: {val_accuracy:.2f}%')

        scheduler.step(avg_val_loss)


        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            no_improve = 0
            torch.save(model.state_dict(), f'best_model_run{run}.pth')
        else:
            no_improve += 1
            if no_improve >= patience:
                print("Early stopping")
                break


    print(f"\nStarting testing for run {run+1}...")
    model.load_state_dict(torch.load(f'best_model_run{run}.pth'))
    model.to(device)
    model.eval()


    test_indices = test_dataset.indices
    test_data = [full_dataset.data[i] for i in test_indices]


    grouped_test_data = defaultdict(list)
    for item in test_data:
        key = (item['model'], item['benchmark'])
        grouped_test_data[key].append(item)


    run_results = []
    for (model_name, benchmark_name), group_items in grouped_test_data.items():
        print(f"\nTesting model: {model_name}, benchmark: {benchmark_name}")
        print(f"Number of test samples: {len(group_items)}")


        group_dataset = VecDataset(group_items)
        group_loader = DataLoader(group_dataset, batch_size=100, shuffle=False)

        test_loss = 0.0
        correct = 0
        total = 0
        all_preds = []
        all_probs = []
        all_labels = []

        with torch.no_grad():
            for features, labels in group_loader:
                features, labels = features.to(device), labels.to(device)
                outputs = model(features)
                test_loss += criterion(outputs, labels).item()

                probs = F.softmax(outputs, dim=1)
                _, predicted = torch.max(outputs.data, 1)

                total += labels.size(0)
                correct += (predicted == labels).sum().item()

                all_preds.extend(predicted.cpu().numpy())
                all_probs.extend(probs.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        avg_test_loss = test_loss / len(group_loader) if len(group_loader) > 0 else 0
        test_accuracy = 100 * correct / total if total > 0 else 0

        try:
            precision = precision_score(all_labels, all_preds)
            recall = recall_score(all_labels, all_preds)
            f1 = f1_score(all_labels, all_preds)
            mcc = matthews_corrcoef(all_labels, all_preds)


            all_probs_array = np.array(all_probs)
            if len(all_probs_array.shape) == 2 and all_probs_array.shape[1] >= 2:
                auc = roc_auc_score(all_labels, all_probs_array[:, 1])
            else:
                print(f"Warning: Invalid probability array shape for {model_name}-{benchmark_name}")
                auc = 0.0

            tn, fp, fn, tp = confusion_matrix(all_labels, all_preds).ravel()

            result = {
                'run': run+1,
                'model': model_name,
                'benchmark': benchmark_name,
                'num_samples': total,
                'test_loss': avg_test_loss,
                'test_accuracy': test_accuracy,
                'precision': precision,
                'recall': recall,
                'f1': f1,
                'mcc': mcc,
                'auc': auc,
                'tp': tp,
                'fp': fp,
                'tn': tn,
                'fn': fn,
                'tpr': tp / (tp + fn) if (tp + fn) > 0 else 0,
                'fpr': fp / (fp + tn) if (fp + tn) > 0 else 0,
            }

            run_results.append(result)

        except Exception as e:
            print(f"Error calculating metrics for {model_name}-{benchmark_name}: {str(e)}")
            continue

    all_run_results.extend(run_results)
    print(f"\nTesting completed for run {run+1}!")


print("\nCalculating median results over 5 runs...")
results_df = pd.DataFrame(all_run_results)


median_results = results_df.groupby(['model', 'benchmark']).median().reset_index()
median_results.drop(columns=['run'], inplace=True)  


cols_order = [
    'model', 'benchmark', 'num_samples', 'test_loss', 'test_accuracy',
    'precision', 'recall', 'f1', 'mcc', 'auc',
    'tp', 'fp', 'tn', 'fn', 'tpr', 'fpr'
]
median_results = median_results[cols_order]


output_file = "experts_all/median_codemi.csv" 
median_results.to_csv(output_file, index=False)
print(f"\nMedian test results saved to {output_file}")

detailed_output_file = "detail_codemi.csv"
results_df.to_csv(detailed_output_file, index=False)
print(f"Detailed test results saved to {detailed_output_file}")

print("\nAll runs completed!")

Using device: cuda

=== Starting Run 1/5 ===
Epoch [1/5], Train Loss: 0.7329, Val Loss: 0.6812, Val Acc: 63.50%
Epoch [2/5], Train Loss: 0.6935, Val Loss: 0.6587, Val Acc: 69.09%
Epoch [3/5], Train Loss: 0.6610, Val Loss: 0.6318, Val Acc: 73.64%
Epoch [4/5], Train Loss: 0.6381, Val Loss: 0.6096, Val Acc: 75.96%
Epoch [5/5], Train Loss: 0.6013, Val Loss: 0.5805, Val Acc: 77.24%

Starting testing for run 1...

Testing model: codegemma-2b, benchmark: mbxp_python
Number of test samples: 334

Testing model: Qwen2.5-Coder-3B-Instruct, benchmark: mbxp_java
Number of test samples: 333

Testing model: codegemma-2b, benchmark: mbxp_java
Number of test samples: 364

Testing model: deepseek-coder-1.3b-instruct, benchmark: mbxp_java
Number of test samples: 347

Testing model: Qwen2.5-Coder-3B-Instruct, benchmark: mbxp_python
Number of test samples: 354

Testing model: codegemma-2b, benchmark: humaneval_java
Number of test samples: 59

Testing model: codegemma-2b, benchmark: humaneval_python
Number 

/home/ubuntu602/anaconda3/envs/CJ_humaneval/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch [3/5], Train Loss: 0.6634, Val Loss: 0.6374, Val Acc: 67.09%
Epoch [4/5], Train Loss: 0.6343, Val Loss: 0.6121, Val Acc: 73.72%
Epoch [5/5], Train Loss: 0.6002, Val Loss: 0.5842, Val Acc: 76.20%

Starting testing for run 2...

Testing model: codegemma-2b, benchmark: mbxp_python
Number of test samples: 334

Testing model: Qwen2.5-Coder-3B-Instruct, benchmark: mbxp_java
Number of test samples: 333

Testing model: codegemma-2b, benchmark: mbxp_java
Number of test samples: 364

Testing model: deepseek-coder-1.3b-instruct, benchmark: mbxp_java
Number of test samples: 347

Testing model: Qwen2.5-Coder-3B-Instruct, benchmark: mbxp_python
Number of test samples: 354

Testing model: codegemma-2b, benchmark: humaneval_java
Number of test samples: 59

Testing model: codegemma-2b, benchmark: humaneval_python
Number of test samples: 45

Testing model: Qwen2.5-Coder-3B-Instruct, benchmark: ncb_python
Number of test samples: 23

Testing model: deepseek-coder-1.3b-instruct, benchmark: mbxp_pytho

/home/ubuntu602/anaconda3/envs/CJ_humaneval/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch [2/5], Train Loss: 0.7109, Val Loss: 0.6596, Val Acc: 56.79%
Epoch [3/5], Train Loss: 0.6697, Val Loss: 0.6262, Val Acc: 68.45%
Epoch [4/5], Train Loss: 0.6349, Val Loss: 0.5920, Val Acc: 75.32%
Epoch [5/5], Train Loss: 0.6079, Val Loss: 0.5747, Val Acc: 76.76%

Starting testing for run 5...

Testing model: codegemma-2b, benchmark: mbxp_python
Number of test samples: 334

Testing model: Qwen2.5-Coder-3B-Instruct, benchmark: mbxp_java
Number of test samples: 333

Testing model: codegemma-2b, benchmark: mbxp_java
Number of test samples: 364

Testing model: deepseek-coder-1.3b-instruct, benchmark: mbxp_java
Number of test samples: 347

Testing model: Qwen2.5-Coder-3B-Instruct, benchmark: mbxp_python
Number of test samples: 354

Testing model: codegemma-2b, benchmark: humaneval_java
Number of test samples: 59

Testing model: codegemma-2b, benchmark: humaneval_python
Number of test samples: 45

Testing model: Qwen2.5-Coder-3B-Instruct, benchmark: ncb_python
Number of test samples: 23


/home/ubuntu602/anaconda3/envs/CJ_humaneval/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
